# Run Calibration Campaign

This notebook starts the Docker Compose services, checks manager/node availability, checks external device endpoints, shows dashboard links, and then runs `experiments/calibration_campaign.py`.

In [ ]:
from __future__ import annotations

import socket
import subprocess
import sys
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

from IPython.display import Markdown, display


def find_project_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "compose.yaml").exists() and (candidate / "experiments").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing compose.yaml")


PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

## 1. Start Docker Compose

Run this cell first. Docker Desktop must already be running.

In [ ]:
def run_command(args: list[str], *, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("$ " + " ".join(args))
    completed = subprocess.run(
        args,
        cwd=PROJECT_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout)
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {' '.join(args)}")
    return completed


run_command(["docker", "compose", "up", "-d"])
time.sleep(5)
run_command(["docker", "compose", "ps"], check=False)

## 2. Check Managers, Nodes, and External Devices

In [ ]:
MANAGER_ENDPOINTS = [
    ("lab_manager / Squid", "http://localhost:8000/"),
    ("event_manager", "http://localhost:8001/"),
    ("experiment_manager", "http://localhost:8002/"),
    ("resource_manager", "http://localhost:8003/"),
    ("data_manager", "http://localhost:8004/"),
    ("workcell_manager", "http://localhost:8005/"),
    ("location_manager", "http://localhost:8006/"),
]

NODE_ENDPOINTS = [
    ("high_viscosity_liquid_weighing_1", "http://localhost:2000/"),
    ("human_node_1", "http://localhost:2001/"),
]

EXTERNAL_DEVICE_ENDPOINTS = [
    ("Raspberry Pi Balance SiLA", "100.124.19.65", 50052),
    ("Raspberry Pi Dispenser ser2net", "100.124.19.65", 2217),
    ("Raspberry Pi Balance ser2net", "100.124.19.65", 2218),
]


def check_http(name: str, url: str, timeout_s: float = 3.0) -> dict[str, str]:
    try:
        req = Request(url, headers={"User-Agent": "auto-calibration-preflight"})
        with urlopen(req, timeout=timeout_s) as response:
            return {"name": name, "target": url, "status": "OK", "detail": f"HTTP {response.status}"}
    except HTTPError as exc:
        if exc.code < 500:
            return {"name": name, "target": url, "status": "OK", "detail": f"HTTP {exc.code}"}
        return {"name": name, "target": url, "status": "FAIL", "detail": f"HTTP {exc.code}"}
    except URLError as exc:
        return {"name": name, "target": url, "status": "FAIL", "detail": str(exc.reason)}
    except Exception as exc:
        return {"name": name, "target": url, "status": "FAIL", "detail": repr(exc)}


def check_tcp(name: str, host: str, port: int, timeout_s: float = 3.0) -> dict[str, str]:
    try:
        with socket.create_connection((host, port), timeout=timeout_s):
            return {"name": name, "target": f"{host}:{port}", "status": "OK", "detail": "TCP connected"}
    except Exception as exc:
        return {"name": name, "target": f"{host}:{port}", "status": "FAIL", "detail": str(exc)}


def markdown_table(rows: list[dict[str, str]]) -> Markdown:
    lines = ["| Check | Target | Status | Detail |", "|---|---|---|---|"]
    for row in rows:
        status = "OK" if row["status"] == "OK" else "FAIL"
        detail = row["detail"].replace("|", "\\|")
        lines.append(f"| {row['name']} | `{row['target']}` | **{status}** | {detail} |")
    return Markdown("\n".join(lines))


manager_results = [check_http(name, url) for name, url in MANAGER_ENDPOINTS]
node_results = [check_http(name, url) for name, url in NODE_ENDPOINTS]
device_results = [check_tcp(name, host, port) for name, host, port in EXTERNAL_DEVICE_ENDPOINTS]
all_results = manager_results + node_results + device_results

display(markdown_table(all_results))

failed = [row for row in all_results if row["status"] != "OK"]
if failed:
    raise RuntimeError(f"Preflight failed: {len(failed)} check(s) failed. Fix these before running the experiment.")

print("Preflight checks passed.")

## 3. Open Dashboards and APIs

In [ ]:
display(Markdown("\n".join([
    "- [Open Squid Dashboard](http://localhost:8000/)",
    "- [Workcell Manager API](http://localhost:8005/docs)",
    "- [Data Manager API](http://localhost:8004/docs)",
    "- [Resource Manager API](http://localhost:8003/docs)",
    "- [High Viscosity Weighing Node API](http://localhost:2000/docs)",
    "- [Human Node API](http://localhost:2001/docs)",
])))

## 4. Run Experiment

Run this cell only after the preflight checks pass and the lab hardware is ready.

In [ ]:
process = subprocess.Popen(
    [sys.executable, "experiments/calibration_campaign.py"],
    cwd=PROJECT_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Experiment failed with exit code {return_code}")

print("Experiment completed successfully.")